In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
class SimpleLSTM(nn.Module):
    """
    간단한 LSTM 모델을 정의하는 클래스입니다.

    Args:
        input_size (int): 입력 크기
        hidden_size (int): 은닉 상태 크기
        output_size (int): 출력 크기
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size

        # batch_first=True 추가
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)  # LSTM 레이어를 정의
        self.fc = nn.Linear(hidden_size, output_size)  # 출력 레이어(분류기)를 정의

    def forward(self, x):
        """
        모델의 순전파를 정의합니다.
        """
        # x.size(0) = batch_size
        h0 = torch.zeros(1, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros(1, x.size(0), self.hidden_size, device=x.device)
        # h0: 초기 은닉 상태를 0으로 초기화
        # LSTM은 이전 시간 단계의 은닉 상태를 현재 시간 단계의 입력으로 사용하기 때문에 초기 은닉 상태가 필요
        # c0: 초기 셀 상태를 0으로 초기화
        # LSTM은 은닉 상태 외에 셀 상태(cell state)라는 내부 상태를 가지고 있음
        # >> 셀 상태는 장기 의존성을 학습하는 데 도움을 줌


        # LSTM에 입력 텐서와 초기 은닉 상태, 셀 상태를 전달합니다.
        # out.shape (batch_size, seq_len, hidden_size)
        out, _ = self.lstm(x, (h0, c0))

        # 시퀀스의 마지막 출력을 분류기에 전달합니다.
        out = self.fc(out[:, -1, :])
        return out

In [5]:
input_size = 10
hidden_size = 20
output_size = 1

# 임의의 훈련 데이터 생성
x_train = torch.randn(100, 5, input_size)
# 100개의 샘플, 각 샘플은 5시간 단계, 입력 특성의 수
y_train = torch.randn(100, 1)
 # 각 샘플에 대한 임의의 출력값

# 임의의 테스트 데이터 생성
x_test = torch.randn(20, 5, input_size)
# 20개의 샘플로 구성된 테스트 데이터
y_test = torch.randn(20, 1)
# 테스트 레이블


In [9]:
# print(x_train)
# print(x_train.shape)
print(y_train)

tensor([[-0.2228],
        [-0.8127],
        [ 0.6729],
        [ 0.6794],
        [ 0.4812],
        [ 1.8354],
        [-0.3046],
        [-2.2029],
        [ 0.4116],
        [-1.5469],
        [ 0.3706],
        [ 0.8920],
        [-1.4523],
        [-0.8301],
        [-1.0875],
        [ 0.4798],
        [-0.2870],
        [-0.3591],
        [-0.9677],
        [-0.5016],
        [-1.1594],
        [-0.5135],
        [-0.0117],
        [-0.8993],
        [-0.2689],
        [-1.4424],
        [-1.0550],
        [-0.9901],
        [-0.0946],
        [-0.4186],
        [-0.4427],
        [-1.4304],
        [-0.0723],
        [ 0.2291],
        [-0.5022],
        [-0.2674],
        [ 0.0401],
        [-0.1695],
        [-2.2378],
        [ 0.8131],
        [-0.1041],
        [ 0.4019],
        [-0.3871],
        [-0.3846],
        [ 0.4984],
        [ 1.2756],
        [-0.4758],
        [-0.4817],
        [-0.8915],
        [-0.2978],
        [ 0.7962],
        [ 0.4257],
        [ 0.

In [10]:
model = SimpleLSTM(input_size, hidden_size, output_size).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [11]:
epochs = 10  # 훈련 에폭 수

for epoch in range(epochs):
    model.train()  # 훈련용 모드 선언
    running_loss = 0.0

    for i in range(len(x_train)):
        inputs = x_train[i].unsqueeze(0).to(device)  # 배치 차원 추가
        # (1, 5, 10)
        labels = y_train[i].unsqueeze(0).to(device)
        # (1, 1)

        outputs = model(inputs)
        # outputs 예측값 (logits)
        loss = criterion(outputs, labels)

        # 반드시 최적화기 초기화 (zero_grad)
        optimizer.zero_grad()
        loss.backward()    # 역전파 (편미분 가능 >> 딥러닝 가능) autograd 기
        optimizer.step()   # 최적화기 이용, w,b (학습파라미터) 수정

        running_loss += loss.item()

    # 평균 loss 확인
    epoch_loss = running_loss / len(x_train)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}')


Epoch [1/10], Loss: 0.8745
Epoch [2/10], Loss: 0.7584
Epoch [3/10], Loss: 0.5860
Epoch [4/10], Loss: 0.3743
Epoch [5/10], Loss: 0.1768
Epoch [6/10], Loss: 0.0791
Epoch [7/10], Loss: 0.1064
Epoch [8/10], Loss: 0.0865
Epoch [9/10], Loss: 0.0529
Epoch [10/10], Loss: 0.0327


In [12]:
model.eval()  # 평가 모드
with torch.no_grad():  # 기울기 계산 비활성화
    correct = 0
    total = 0

    for i in range(len(x_test)):
        inputs = x_test[i].unsqueeze(0).to(device)
        labels = y_test[i].unsqueeze(0).to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        print(f'Test Sample: {i+1}, Loss: {loss.item():.4f}')

Test Sample: 1, Loss: 0.7868
Test Sample: 2, Loss: 1.2257
Test Sample: 3, Loss: 2.2728
Test Sample: 4, Loss: 0.3691
Test Sample: 5, Loss: 0.0137
Test Sample: 6, Loss: 1.4194
Test Sample: 7, Loss: 2.5764
Test Sample: 8, Loss: 0.2841
Test Sample: 9, Loss: 0.5312
Test Sample: 10, Loss: 0.0009
Test Sample: 11, Loss: 0.3049
Test Sample: 12, Loss: 2.5045
Test Sample: 13, Loss: 1.8377
Test Sample: 14, Loss: 0.2870
Test Sample: 15, Loss: 4.2884
Test Sample: 16, Loss: 0.8192
Test Sample: 17, Loss: 3.7895
Test Sample: 18, Loss: 0.6549
Test Sample: 19, Loss: 0.2044
Test Sample: 20, Loss: 3.0037
